# Cookie Cats A/B Testi — 05: Sonuçlar ve Karar

Bu notebook önceki dört notebook'un bulgularını tek yerde topluyor ve ürün kararını veriyor. Rakamlar aşağıdaki hücrede temiz veriden **yeniden hesaplanıyor**; elle yazılmış bir sayı yok.

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import norm, mannwhitneyu, chisquare
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

pd.set_option("display.float_format", "{:.4f}".format)
ALPHA, POWER = 0.05, 0.80
Z_A, Z_B = norm.ppf(1 - ALPHA / 2), norm.ppf(POWER)

raw = pd.read_csv("../data/cookie_cats.csv")
df = pd.read_csv("../data/cookie_cats_clean.csv")
a, b = df[df["version"] == "gate_30"], df[df["version"] == "gate_40"]

rows = []
for m in ["retention_7", "retention_1"]:
    x, n = np.array([b[m].sum(), a[m].sum()]), np.array([len(b), len(a)])
    p40, p30 = x / n
    _, p = proportions_ztest(x, n)
    lo, hi = confint_proportions_2indep(x[0], n[0], x[1], n[1], method="newcomb", compare="diff")
    se = np.sqrt(p30 * (1 - p30) * (1 / n[0] + 1 / n[1]))
    rows.append({"metrik": m, "gate_30": p30, "gate_40": p40, "fark_puan": (p40 - p30) * 100,
                 "CI_alt": lo * 100, "CI_ust": hi * 100, "göreli_%": (p40 / p30 - 1) * 100, "p": p,
                 "MDE_puan": (Z_A + Z_B) * se * 100})
_, p_mw = mannwhitneyu(b["sum_gamerounds"], a["sum_gamerounds"])
_, p_srm = chisquare(raw["version"].value_counts().values)

results = pd.DataFrame(rows).set_index("metrik")
display(results)
print(f"Round sayısı: medyan gate_30 = {a['sum_gamerounds'].median():.0f}, gate_40 = {b['sum_gamerounds'].median():.0f}, Mann-Whitney p = {p_mw:.4f}")
print(f"SRM: p = {p_srm:.4f}")

,gate_30,gate_40,fark_puan,CI_alt,CI_ust,göreli_%,p,MDE_puan
metrik,,,,,,,,
retention_7,0.1902,0.1820,-0.8183,-1.3264,-0.3103,-4.3028,0.0016,0.7322
retention_1,0.4482,0.4423,-0.5915,-1.2402,0.0572,-1.3198,0.0739,0.9279


Round sayısı: medyan gate_30 = 17, gate_40 = 16, Mann-Whitney p = 0.0509
SRM: p = 0.0086


## 1. Test sonuçları özeti (notebook 03)

- **Ana metrik (retention_7):** gate_40, gate_30'a göre **0.82 puan** (göreli **%4.3**) daha düşük: %19.02 → %18.20. %95 güven aralığı **[−1.33, −0.31]** puan, **p = 0.0016**. → **İstatistiksel olarak anlamlı.** Güven aralığının tamamı negatif tarafta; bootstrap (5.000 tekrar) aynı aralığı verdi, Holm düzeltmesinden sonra da anlamlı kalıyor (p = 0.005).
- **retention_1:** gate_40 **0.59 puan** daha düşük (%44.82 → %44.23), %95 CI **[−1.24, +0.06]**, **p = 0.074**. → İstatistiksel olarak anlamlı **değil**; ancak yön retention_7 ile aynı.
- **Round sayısı (koruma metriği):** Medyan 17'ye karşı 16; Mann-Whitney **p = 0.051**, üstünlük olasılığı **0.496** (0.5 = fark yok). → Pratikte **fark yok**: kapının yeri oyuncuların ne kadar oynadığını değiştirmemiş, sadece geri dönüp dönmediklerini etkilemiş.
- **Sınırlamalar:**
  - **SRM** (p ≈ 0.009): gruplar %50.4 / %49.6. Atama mekanizması bilinmediği için nedeni belirlenemedi.
  - **Level bilgisi yok:** kapı level'a göre uygulanıyor ama veride sadece round sayısı var; kimin kapıya ulaştığı doğrudan gözlenemiyor.
  - **Gelir verisi yok:** kapının satın alma davranışına etkisi ölçülemiyor.

## 2. Güç analizi özeti (notebook 04)

- Bu deney (grup başına ~45 bin oyuncu), retention_7'de en az **0.73 puanlık** (göreli **%3.9**) bir farkı %80 güçle yakalayabilirdi. Gözlenen fark **0.82 puan** → deney bu fark için **yeterli** (güç ≈ %88). retention_7 bulgusu güvenilir.
- retention_1'de MDE **0.93 puan**, gözlenen fark **0.59 puan** → bu büyüklükte bir etkiyi yakalama olasılığı yalnızca **~%43**. "Anlamlı değil" sonucu **"etki yok"** değil, **"deney bu büyüklükte bir etkiyi yakalayacak kadar hassas değildi"** olarak yorumlanmalı. Güvenilir test için grup başına ~111 bin oyuncu (mevcudun ~2.5 katı) gerekirdi.
- Simülasyon formülleri doğruladı: etki yokken yanlış alarm oranı %4.9 (beklenen %5); gözlenen farklar için simüle edilen güç %88.6 ve %44.7 (formül: %87.9 ve %43.1).
- **Bir sonraki deneyi tasarlasaydım:**
  1. **Önce** iş açısından önemli en küçük farkı (MDE) belirler, örneklem büyüklüğünü ve süreyi buna göre hesaplardım; analiz planını deney başlamadan yazardım.
  2. Atama ve loglama sistemini deney başında **SRM kontrolüyle** izlerdim; dengesizlik çıkarsa deneyi durdurup nedenini bulurdum.
  3. **Level** (kapıya ulaşıldı mı) ve **gelir/satın alma** verisini toplardım; böylece hem kapıya maruz kalan oyunculardaki etkiyi hem de retention ile gelir arasındaki dengeyi ölçebilirdim.

## 3. Karar ve öneri

**Öneri: Kapıyı level 30'da tutmak.** Kapı zaten level 30'da (kontrol grubu); bu deneyin sonucu kapıyı 40'a taşımamak yönünde.

**1. İstatistiksel anlamlılık.** Ana metrik retention_7'de fark anlamlı (p = 0.0016) ve sonuç bootstrap ile Holm düzeltmesine dayanıklı. retention_1'de fark anlamlı değil; ancak 1. günde kapıya ulaşan oyuncu az olduğu için etki sulanıyor ve deney bu büyüklükte bir etkiyi yakalayacak kadar güçlü değildi (güç ≈ %43). Bu "etki yok" değil, "yeterli kanıt yok" demek.

**2. Pratik (iş) anlamlılığı.** Göreli %4.3'lük bir 7. gün retention düşüşü bir mobil oyun için küçük değil. Test havuzunda gate_40 grubunda ~370 oyuncu daha az geri döndü; günde 100 bin yeni oyuncu varsayımıyla bu, her gün ~820 (güven aralığına göre 310–1.330) oyuncu demek. 7. günde kaybedilen oyuncu sonraki haftalarda da yoktur; bu kayıp aktif oyuncu sayısına ve gelire zamanla birikerek yansır.

**3. Ürün ekibi için tek cümle.** *"Kapıyı 40'a taşırsak 7 gün sonra oyuna dönen oyuncu oranı göreli olarak ~%4 düşüyor; günde 100 bin yeni oyuncuda bu her gün yüzlerce kaybedilen oyuncu demek. Kapıyı 30'da tutmayı öneriyorum."*

**4. Riskler ve sınırlamalar.**
- **Gelir verisi yok:** gate_40 retention'ı düşürürken geliri artırıyor olabilir; bu veriyle ölçülemez. Ancak deneyin amacı kapı konumunun retention'a etkisini ölçmekti ve riskler asimetrik: 30'da kalmak mevcut durumu korur, 40'a taşımak ise kanıtlanmış bir retention kaybı getirir.
- **SRM:** Nedeni bilinmeyen küçük bir örneklem dengesizliği var; sonuçlara güven bu oranda sınırlı ve nedeni araştırılmalı.
- **Level bilgisi yok:** Kimin kapıya ulaştığı doğrudan gözlenemiyor.

**5. Sonraki deneyler.**
- **Erken oyun deneyi:** Medyan oyuncu 14 günde yalnızca 16 round oynuyor; yani oyuncuların çoğu kapıyı hiç görmeden bırakıyor ve asıl kayıp kapıdan önce oluşuyor. Gate_30'un daha iyi çalışmasının olası bir açıklaması, erken gelen zorunlu molanın oyuncunun oyundan sıkılmasını geciktirmesi. Bunu test etmek için level 10–15 aralığına ödeme seçeneği olmayan kısa bir mola noktası ekleyip erken oyun retention'ını ölçerdim.
- **Gelir dahil kapı deneyi:** Gelir önemliyse kapı konumu, oyuncu başına gelir / LTV ana metrik ve retention koruma metriği olacak şekilde, level verisiyle ve önceden hesaplanmış örneklem büyüklüğüyle tekrar test edilmeli.